# LSTM Baseline - CLINC150

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import time
from collections import Counter
from data_loader import load_clinc

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = True
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

cuda
1525 / 620 / 1100, 151 classes (sample)


In [2]:
class Vocabulary:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.min_freq = min_freq

    def build_vocab(self, texts):
        counts = Counter()
        for t in texts:
            counts.update(t.lower().split())
        idx = 2
        for word, cnt in counts.items():
            if cnt >= self.min_freq:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

    def encode(self, text):
        return [self.word2idx.get(w, 1) for w in text.lower().split()]

    def __len__(self):
        return len(self.word2idx)


vocab = Vocabulary(min_freq=2)
vocab.build_vocab(train_texts)
vocab_size = len(vocab)
print(f"Vocab: {vocab_size}")

Vocab: 876


In [3]:
class TextSequenceDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.LongTensor(self.vocab.encode(self.texts[idx])), torch.LongTensor([self.labels[idx]])


def collate_batch(batch):
    seqs, labels = zip(*batch)
    return pad_sequence(seqs, batch_first=True, padding_value=0), torch.LongTensor([len(s) for s in seqs]), torch.cat(labels)


BATCH_SIZE = 64

train_loader = DataLoader(TextSequenceDataset(train_texts, train_labels, vocab), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(TextSequenceDataset(val_texts, val_labels, vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(TextSequenceDataset(test_texts, test_labels, vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

In [4]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0,
                           bidirectional=bidirectional)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), num_classes)

    def forward(self, sequences, lengths):
        embedded = self.embedding(sequences)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        if self.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        return self.fc(self.dropout(hidden))


EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3
BIDIRECTIONAL = True

model = LSTMClassifier(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, num_classes,
                       NUM_LAYERS, DROPOUT, BIDIRECTIONAL).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} params")

2,557,079 params


In [5]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for seqs, lengths, labels in loader:
        seqs, lengths, labels = seqs.to(device), lengths.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(seqs, lengths)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100 * correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for seqs, lengths, labels in loader:
            seqs, lengths, labels = seqs.to(device), lengths.to(device), labels.to(device)
            outputs = model(seqs, lengths)
            total_loss += criterion(outputs, labels).item()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), np.array(all_preds), np.array(all_labels)

In [6]:
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
PATIENCE = 5

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_val_f1, best_model_state, patience_counter, best_epoch = 0, None, 0, 0
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_preds, vl_true = evaluate(model, val_loader, criterion)
    vl_f1 = f1_score(vl_true, vl_preds, average='macro', zero_division=0)
    vl_acc = accuracy_score(vl_true, vl_preds)

    lr = optimizer.param_groups[0]['lr']
    scheduler.step(vl_f1)

    mark = ""
    if vl_f1 > best_val_f1:
        best_val_f1, best_model_state, best_epoch, patience_counter = vl_f1, model.state_dict().copy(), epoch + 1, 0
        mark = " *"
    else:
        patience_counter += 1

    print(f"{epoch+1:2d}/{NUM_EPOCHS}  tr_loss={tr_loss:.4f} tr_acc={tr_acc:.1f}%  vl_loss={vl_loss:.4f} vl_acc={vl_acc*100:.1f}% vl_f1={vl_f1:.4f} lr={lr:.6f}{mark}")

    if patience_counter >= PATIENCE:
        print(f"Early stopping. Best epoch: {best_epoch}, F1: {best_val_f1:.4f}")
        break

training_time = time.time() - start_time
print(f"Time: {training_time:.1f}s")

 1/50  tr_loss=4.9568 tr_acc=8.5%  vl_loss=4.8336 vl_acc=13.4% vl_f1=0.1047 lr=0.001000 *
 2/50  tr_loss=4.1279 tr_acc=23.8%  vl_loss=3.9066 vl_acc=18.4% vl_f1=0.1578 lr=0.001000 *
 3/50  tr_loss=2.5246 tr_acc=54.0%  vl_loss=3.1712 vl_acc=31.1% vl_f1=0.2873 lr=0.001000 *
 4/50  tr_loss=1.3475 tr_acc=78.9%  vl_loss=2.7167 vl_acc=38.7% vl_f1=0.3679 lr=0.001000 *
 5/50  tr_loss=0.6871 tr_acc=90.2%  vl_loss=2.4539 vl_acc=43.7% vl_f1=0.4272 lr=0.001000 *
 6/50  tr_loss=0.3545 tr_acc=95.3%  vl_loss=2.3249 vl_acc=46.0% vl_f1=0.4446 lr=0.001000 *
 7/50  tr_loss=0.1989 tr_acc=98.0%  vl_loss=2.3185 vl_acc=47.9% vl_f1=0.4625 lr=0.001000 *
 8/50  tr_loss=0.1242 tr_acc=98.6%  vl_loss=2.3538 vl_acc=47.7% vl_f1=0.4595 lr=0.001000
 9/50  tr_loss=0.0842 tr_acc=99.1%  vl_loss=2.3901 vl_acc=47.6% vl_f1=0.4660 lr=0.001000 *
10/50  tr_loss=0.0598 tr_acc=99.4%  vl_loss=2.3773 vl_acc=47.1% vl_f1=0.4599 lr=0.001000
11/50  tr_loss=0.0484 tr_acc=99.5%  vl_loss=2.3682 vl_acc=49.0% vl_f1=0.4790 lr=0.001000 *
12/5

In [7]:
model.load_state_dict(best_model_state)
_, test_preds, test_true = evaluate(model, test_loader, criterion)

acc = accuracy_score(test_true, test_preds)
p = precision_score(test_true, test_preds, average='macro', zero_division=0)
r = recall_score(test_true, test_preds, average='macro', zero_division=0)
f1_m = f1_score(test_true, test_preds, average='macro', zero_division=0)
f1_w = f1_score(test_true, test_preds, average='weighted', zero_division=0)

print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"F1 macro:   {f1_m:.4f}")
print(f"F1 weighted:{f1_w:.4f}")

Accuracy:   0.4891
Precision:  0.5230
Recall:     0.5210
F1 macro:   0.4821
F1 weighted:0.4732


In [8]:
results = {
    "model_type": "LSTM",
    "optimization": "baseline",
    "best_epoch": best_epoch,
    "training_time_seconds": round(training_time, 2),
    "test_metrics": {
        "accuracy": round(acc, 4),
        "macro_precision": round(p, 4),
        "macro_recall": round(r, 4),
        "macro_f1": round(f1_m, 4),
        "weighted_f1": round(f1_w, 4)
    },
    "hyperparameters": {
        "vocab_min_freq": 2,
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "bidirectional": BIDIRECTIONAL,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "max_epochs": NUM_EPOCHS,
        "early_stopping_patience": PATIENCE
    }
}

with open('results/results_lstm_baseline.json', 'w') as f:
    json.dump(results, f, indent=4)

print("Saved: results/results_lstm_baseline.json")

Saved: results/results_lstm_baseline.json
